In [11]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
scaled_train = pd.read_csv('../data/processed/scaled_train.csv')
scaled_val = pd.read_csv('../data/processed/scaled_val.csv')

In [13]:
X, y = scaled_train.drop(columns=["result"]), scaled_train["result"]

In [14]:
X_val = scaled_val.drop(columns=["result"])
y_val = scaled_val["result"]

In [15]:
X = pd.concat([X, X_val], ignore_index=True)
y = pd.concat([y, y_val], ignore_index=True)

In [ ]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Base models
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    random_state=42,
    eval_metric="mlogloss"
)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

svc = SVC(
    probability=True,
    random_state=42
)

# Ensemble
ensemble = VotingClassifier(
    estimators=[
        ("xgb", xgb),
        ("lr", lr),
        ("svc", svc)
    ],
    voting="soft"      # Uses probabilities
)

# Parameters to tune
param_grid = {
    "xgb__n_estimators": [300, 500],
    "xgb__learning_rate": [0.03, 0.05],
    "xgb__max_depth": [3, 5],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0],

    "lr__C": [0.1, 1, 10],

    "svc__C": [0.1, 1, 10],
    "svc__gamma": ["scale", "auto"]
}

grid = GridSearchCV(
    ensemble,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X, y)

best_model = grid.best_estimator_

print("Best Parameters:")
print(grid.best_params_)

print("Best CV Accuracy:")
print(grid.best_score_)

# Predictions
y_pred = best_model.predict(X)

probs = best_model.predict_proba(X)

print(probs.shape)
print(probs[:5])

print("Train Accuracy:", accuracy_score(y, y_pred))


/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/ziadsamer/Projects/nations-matche

Best Parameters:
{'lr__C': 0.1, 'svc__C': 0.1, 'svc__gamma': 'auto', 'xgb__colsample_bytree': 1.0, 'xgb__learning_rate': 0.03, 'xgb__max_depth': 3, 'xgb__n_estimators': 300, 'xgb__subsample': 0.8}
Best CV Accuracy:
0.5187018489984592
(883, 3)
[[0.23139609 0.61629909 0.15230482]
 [0.19284801 0.11380491 0.69334708]
 [0.22200529 0.37792117 0.40007355]
 [0.22228572 0.2521058  0.52560846]
 [0.45680343 0.31147998 0.23171662]]
Train Accuracy: 0.6613816534541337
Validation Accuracy: 0.672316384180791


/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [17]:
import joblib
joblib.dump(best_model, "../models/best_model.pkl")

['../models/best_model.pkl']

In [19]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.26      0.41       230
           1       0.63      0.86      0.73       370
           2       0.66      0.72      0.69       283

    accuracy                           0.66       883
   macro avg       0.75      0.62      0.61       883
weighted avg       0.73      0.66      0.63       883



In [187]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

                     feature  importance
19              ranking_diff    0.091084
10     away_relative_shots_2    0.051561
12              away_ranking    0.049040
3       home_shots_against_1    0.047008
16     home_stadium_temp_avg    0.046404
18   away_stadium_wind_speed    0.045898
11              home_ranking    0.045372
20   stadium_temperature_avg    0.043806
17     away_stadium_temp_avg    0.043393
14           away_wind_speed    0.041650
0   away_stadium_distance_km    0.041327
1                 home_fix_1    0.040689
8                 away_fix_2    0.040178
4       home_shots_against_2    0.039650
13           home_wind_speed    0.039264
5      home_relative_shots_1    0.039036
21      home_temperature_avg    0.038774
15        stadium_wind_speed    0.037967
22      away_temperature_avg    0.037023
6      home_relative_shots_2    0.036994
7                 away_fix_1    0.036980
2                 home_fix_2    0.033981
9      away_relative_shots_1    0.032922


In [188]:
import joblib
joblib.dump(model, '../models/xgboost.pkl')

['../models/xgboost.pkl']

# Inference